# Data Engineering Case Study
## End-to-end data lake in Databricks (Bronze → Silver → Gold)

**Submission:** Executable IPython/Jupyter notebook, including the source ingestion, schema and table **DDL**, transformations, validation, and documentation of the `top_item` datamart.

**Technology:** Databricks Free Edition · PySpark DataFrames · Spark SQL · Unity Catalog · Delta Lake

### Business objective
Create a `top_item` datamart containing, **for every item and every year covered by item-view events**:

1. the total number of genuine item-view events in that year;
2. the annual rank based on that number of views;
3. the platform responsible for the most views of that item in that year.

### Architecture

| Layer | Input | Output | Data grain and rules |
|:--|:--|:--|:--|
| **Bronze** | Two source CSV files | `bronze.event`, `bronze.item` | One row per source CSV record; all columns **STRING**; preserve source attributes and records. |
| **Silver** | Bronze | `silver.event`, `silver.item` | Event fact: **one row per `event_id`** after flattening JSON and pivoting parameter records; item dimension: one row per item; typed, `snake_case` fields. Event fact physically partitioned by year. |
| **Gold** | Silver | `gold.top_item` | One row per **(`item_id`, `year`)**; views, `dense_rank`, most-used platform, and useful item descriptions. |

**Reproducibility:** Execute every cell from top to bottom. Rerunning the notebook safely replaces data in its three managed Delta table layers (`INSERT OVERWRITE`); it does not append duplicate records.

**Assumptions:** The catalog is `workspace` and both source CSV files have been uploaded to a Unity Catalog volume. If your catalog/volume differs, change only the configuration cell below. Databricks Free Edition may block direct HTTP/S3 reads from notebooks; uploading the files to a volume avoids that restriction.

## 1. Source inventory and business interpretation

Public data locations  can be found in the assignment.

Use **Catalog → volume → Upload files** in Databricks to upload `event.csv` and `item.csv` before starting. No Databricks cluster management is required on Free Edition; attach the notebook to available **Serverless** compute.

| Source | Raw schema | Rows in supplied files | Interpretation |
|:--|:--|--:|:--|
| `item.csv` | `adjective`, `category`, `created_at`, `id`, `modifier`, `name`, `price` | 2,198 | Reference/dimension data describing catalog items. |
| `event.csv` | `event_id`, `event_time`, `user_id`, `event.payload` | 853,640 | Parameter-level records of user activity. `event.payload` contains JSON encoded as a CSV string. |

### Key observation: rows and events are not the same thing

In `event.csv`, the same `event_id` can appear on **multiple rows** with different `parameter_name` / `parameter_value` combinations. For example, one view event has both an `item_id` parameter and a `referrer` parameter. Thus **853,640 source records correspond to 428,364 logical events**, including **262,786 unique `view_item` events**.

Counting raw source rows would **double-count** views. Bronze must nevertheless keep all original parameter rows, while Silver converts these into one event-level record. Other event types (`test_assignment` and `view_user_profile`) are retained in Silver even though they are not relevant to the Gold metric.

## 2. Runtime configuration

Only this cell usually needs editing. Set the catalog and paths to match the files you uploaded in Step 2 of the exercise. The `workspace` catalog is the default assumed in this submission.

Paths should point to files under `/Volumes/<catalog>/<schema>/<volume>/...`; in the Databricks Catalog explorer, use **Copy path** on a file to get the correct value. If you created `workspace.default.raw_csv_input`, the defaults below should already work.

**Caution:** The notebook overwrites the named Bronze, Silver and Gold tables each time it runs. Use a dedicated catalog/schema if those table names contain important data in your workspace.

In [0]:
# ---------- USER CONFIGURATION (edit paths here if needed) ----------
CATALOG = "workspace"
EVENT_PATH = "/Volumes/workspace/default/raw_csv_input/event.csv"
ITEM_PATH  = "/Volumes/workspace/default/raw_csv_input/item.csv"

# Three-part Unity Catalog names; all reads/writes below use these variables.
BRONZE_EVENT = f"{CATALOG}.bronze.event"
BRONZE_ITEM  = f"{CATALOG}.bronze.item"
SILVER_EVENT = f"{CATALOG}.silver.event"
SILVER_ITEM  = f"{CATALOG}.silver.item"
GOLD_TOP_ITEM = f"{CATALOG}.gold.top_item"

print("Source paths:", EVENT_PATH, ITEM_PATH, sep="\n  ")
print("Target Gold table:", GOLD_TOP_ITEM)

Source paths:
  /Volumes/workspace/default/raw_csv_input/event.csv
  /Volumes/workspace/default/raw_csv_input/item.csv
Target Gold table: workspace.gold.top_item


## 3. Bronze — retain the raw source as STRING

**Requirement:** One managed Delta table per CSV, same source columns, same raw values, **every attribute STRING**; no business transformations, deduplication, column renaming or JSON extraction.

The CSV parser must correctly handle doubled quotation marks inside quoted CSV fields (the source JSON uses this form). `.option("escape", '"')` is necessary; without it, JSON functions can later produce all-null results.

`inferSchema=false` is deliberate: Bronze is a replayable representation of the original records, not a cleaned business layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Read the raw CSVs; the CSV parser handles file quoting, but no business
# attributes are renamed or converted. inferSchema=False makes all fields STRING.
source_events = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .csv(EVENT_PATH)
)

source_items = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .csv(ITEM_PATH)
)

# Confirm file/header expectations before writing permanent tables.
assert source_events.columns == ["event_id", "event_time", "user_id", "event.payload"]
assert source_items.columns == ["adjective", "category", "created_at", "id", "modifier", "name", "price"]
assert all(isinstance(f.dataType, T.StringType) for f in source_events.schema.fields)
assert all(isinstance(f.dataType, T.StringType) for f in source_items.schema.fields)

print("Raw event rows:", source_events.count())
print("Raw item rows:", source_items.count())
source_events.printSchema()
source_items.printSchema()

Raw event rows: 853640
Raw item rows: 2198
root
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event.payload: string (nullable = true)

root
 |-- adjective: string (nullable = true)
 |-- category: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price: string (nullable = true)



### 3.1 Bronze DDL — create schemas and raw tables

`CREATE SCHEMA` defines the logical namespace; `CREATE TABLE ... USING DELTA` explicitly specifies storage format and column data types. Backticks are required around `` `event.payload` `` because its literal column name contains a period.

Tables are managed by Unity Catalog. No `LOCATION` clause is needed because Databricks manages the underlying storage.

In [0]:
# Schema DDL: create one database schema per medallion layer.
for layer in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{layer}")

# Table DDL: every original event column is STRING, including the dotted name.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_EVENT} (
    event_id STRING,
    event_time STRING,
    user_id STRING,
    `event.payload` STRING
)
USING DELTA
""")

# Table DDL: original item attributes retained with STRING types.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_ITEM} (
    adjective STRING,
    category STRING,
    created_at STRING,
    id STRING,
    modifier STRING,
    name STRING,
    price STRING
)
USING DELTA
""")

print("Bronze schemas and tables are ready.")

Bronze schemas and tables are ready.


### 3.2 Bronze ingestion (idempotent DML)

`INSERT OVERWRITE` replaces the previous table contents on rerun, making the notebook repeatable. These statements **do not** clean or reshape the source. Column order is explicit so the CSV and table schemas align.

In [0]:
# Temporary views let SQL load the in-memory CSV DataFrames into Delta tables.
source_events.createOrReplaceTempView("stg_source_events")
source_items.createOrReplaceTempView("stg_source_items")

spark.sql(f"""
INSERT OVERWRITE TABLE {BRONZE_EVENT}
SELECT event_id, event_time, user_id, `event.payload`
FROM stg_source_events
""")

spark.sql(f"""
INSERT OVERWRITE TABLE {BRONZE_ITEM}
SELECT adjective, category, created_at, id, modifier, name, price
FROM stg_source_items
""")

bronze_events = spark.table(BRONZE_EVENT)
bronze_items = spark.table(BRONZE_ITEM)

# Required Bronze verification: raw record counts and all-STRING attribute types.
assert bronze_events.count() == 853_640
assert bronze_items.count() == 2_198
assert all(isinstance(f.dataType, T.StringType) for f in bronze_events.schema.fields)
assert all(isinstance(f.dataType, T.StringType) for f in bronze_items.schema.fields)
print("Bronze validated: 853,640 event parameter records and 2,198 items.")

Bronze validated: 853,640 event parameter records and 2,198 items.


## 4. Source profiling / data quality assumptions

Before reshaping a fact table, check its **grain** and event parameters. We expect:

| Logical event | Parameter names present |
|:--|:--|
| `view_item` | `item_id`, `referrer` |
| `test_assignment` | `test_id`, `test_assignment` |
| `view_user_profile` | `viewed_user_id` |

Each `event_id` should have consistent timestamp, user, name and platform across its parameter rows. Every (`event_id`, `parameter_name`) pair should occur **once**. These constraints make `first(parameter_value)` within a pivot safe and unambiguous.

Rather than assuming JSON is already a structured column, parse it explicitly using Spark's `from_json` with a declared schema.

In [0]:
# JSON payload schema; parameter values are still strings at this stage.
payload_schema = T.StructType([
    T.StructField("event_name", T.StringType(), True),
    T.StructField("platform", T.StringType(), True),
    T.StructField("parameter_name", T.StringType(), True),
    T.StructField("parameter_value", T.StringType(), True),
])

# Parse and flatten the JSON but keep event_time and user_id as source strings
# for now; typing those columns is part of the Silver transformation.
parsed_events = (
    bronze_events
    .withColumn("payload", F.from_json(F.col("`event.payload`"), payload_schema))
    .select(
        "event_id", "event_time", "user_id",
        F.col("payload.event_name").alias("event_name"),
        F.col("payload.platform").alias("platform"),
        F.col("payload.parameter_name").alias("parameter_name"),
        F.col("payload.parameter_value").alias("parameter_value"),
    )
)

# Any parsing failure here means the source cannot safely enter Silver.
assert parsed_events.filter(F.col("event_name").isNull()).count() == 0
assert parsed_events.filter(F.col("parameter_name").isNull()).count() == 0

# Each event parameter is unique in this provided dataset.
duplicate_parameters = (
    parsed_events
    .groupBy("event_id", "parameter_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert duplicate_parameters == 0, "Unexpected repeated parameters: investigate pivot conflict policy."

# Metadata must be consistent for all parameter rows of the same event.
conflicting_metadata = (
    parsed_events
    .select("event_id", "event_time", "user_id", "event_name", "platform")
    .distinct()
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert conflicting_metadata == 0, "Inconsistent event metadata across parameter rows."

print("Event types and source record counts:")
display(parsed_events.groupBy("event_name").count().orderBy("event_name"))
print("Event parameter names:")
display(parsed_events.groupBy("event_name", "parameter_name").count().orderBy("event_name", "parameter_name"))

Event types and source record counts:


event_name,count
test_assignment,324980
view_item,525572
view_user_profile,3088


Event parameter names:


event_name,parameter_name,count
test_assignment,test_assignment,162490
test_assignment,test_id,162490
view_item,item_id,262786
view_item,referrer,262786
view_user_profile,viewed_user_id,3088


## 5. Silver — clean the item dimension

Silver uses `snake_case` names and converts business fields into their proper types. We rename `id` to `item_id` and `name` to `item_name` to make joins easier.

| Column | Type | Rule |
|:--|:--|:--|
| `item_id` | BIGINT | Raw IDs look like `2512.0`; remove the terminal `.0` before integer cast. |
| `item_name`, `category` | STRING | Description fields; preserve their text. |
| `adjective`, `modifier` | STRING (nullable) | Convert empty strings to SQL `NULL`. |
| `created_at` | TIMESTAMP | Parse `yyyy-MM-dd HH:mm:ss`. Original data has no timezone offset. |
| `price` | DECIMAL(12,2) | Monetary value rounded to two decimal places, not represented as a binary float. |

The item table is a **dimension**, not an event fact, so a manual year partition is unnecessary.

In [0]:
# Helper used only in Silver: represent a blank optional text attribute as NULL.
def null_if_blank(column_name):
    field = F.col(column_name)
    return F.when(F.trim(field) == "", F.lit(None).cast("string")).otherwise(field)

silver_items_df = bronze_items.select(
    F.regexp_replace(F.trim(F.col("id")), r"\.0$", "").cast("bigint").alias("item_id"),
    F.col("name").alias("item_name"),
    F.col("category"),
    null_if_blank("adjective").alias("adjective"),
    null_if_blank("modifier").alias("modifier"),
    F.to_timestamp("created_at", "yyyy-MM-dd HH:mm:ss").alias("created_at"),
    F.col("price").cast("decimal(12,2)").alias("price"),
)

# Fail early if type conversion loses data rather than writing invalid rows.
assert silver_items_df.filter(F.col("item_id").isNull()).count() == 0
assert silver_items_df.filter(F.col("created_at").isNull()).count() == 0
assert silver_items_df.filter(F.col("price").isNull()).count() == 0
assert silver_items_df.select("item_id").distinct().count() == 2_198

silver_items_df.printSchema()
display(silver_items_df.limit(5))

root
 |-- item_id: long (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- adjective: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)



item_id,item_name,category,adjective,modifier,created_at,price
2512,fuzzy contraption carrying_case,contraption,fuzzy,carrying_case,2014-01-15T21:36:09.000Z,150.00
482,instrument refill,instrument,null,refill,2013-05-14T05:20:50.000Z,35.20
2446,industrial-strength module,module,industrial-strength,null,2014-02-04T19:28:32.000Z,300.00
1312,digital tool carrying_case,tool,digital,carrying_case,2013-02-25T12:23:18.000Z,16.50
3556,miniature device cleaner,device,miniature,cleaner,2013-08-05T17:20:45.000Z,16.50


### 5.1 Silver item DDL + load

The DDL makes the expected target contract explicit, rather than relying only on Spark's inferred DataFrame schema. `INSERT OVERWRITE` safely rebuilds the table from its Bronze source.

In [0]:
# Explicit Delta table definition (DDL).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_ITEM} (
    item_id BIGINT,
    item_name STRING,
    category STRING,
    adjective STRING,
    modifier STRING,
    created_at TIMESTAMP,
    price DECIMAL(12,2)
)
USING DELTA
""")

silver_items_df.createOrReplaceTempView("stg_silver_items")

# Load the typed item rows in the same order as the declared schema (DML).
spark.sql(f"""
INSERT OVERWRITE TABLE {SILVER_ITEM}
SELECT item_id, item_name, category, adjective, modifier, created_at, price
FROM stg_silver_items
""")

print("Silver item table rows:", spark.table(SILVER_ITEM).count())

Silver item table rows: 2198


## 6. Silver — normalize the event fact table

**Transformation:** group all parameter rows with the same event metadata, then pivot `parameter_name` into typed columns. This gives **one row per logical `event_id`** instead of one row per event parameter.

| Silver column | Type | Explanation |
|:--|:--|:--|
| `event_id` | STRING | Stable unique event identifier. |
| `event_time` | TIMESTAMP | Timestamp parsed from the source. |
| `user_id` | BIGINT | Raw format uses a `.0` suffix. |
| `event_name`, `platform` | STRING | Event classification and client/platform. |
| `item_id` | BIGINT | Populated for `view_item` events. |
| `referrer` | STRING | Populated for `view_item` events. |
| `viewed_user_id` | BIGINT | Populated for `view_user_profile`. |
| `test_id`, `test_assignment` | INT | Populated for `test_assignment`. |
| `event_year` | INT | Derived from `event_time`; physical fact-table partition. |

A missing value for a parameter that does **not apply** to a given event type is expected and represented as `NULL`.

**Why pivot instead of deduplicate?** Two source rows with the same event ID can contain different parameters. Dropping one would discard information; pivot preserves both values and establishes a correct event-level grain.

In [0]:
# Parse source strings into typed metadata before grouping.
event_parameters = (
    parsed_events
    .select(
        "event_id",
        F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss").alias("event_time"),
        F.regexp_replace(F.trim(F.col("user_id")), r"\.0$", "").cast("bigint").alias("user_id"),
        "event_name", "platform", "parameter_name", "parameter_value",
    )
)

# Validate the main metadata casts; malformed dates/IDs should not be silently kept.
assert event_parameters.filter(F.col("event_time").isNull()).count() == 0
assert event_parameters.filter(F.col("user_id").isNull()).count() == 0

# Enumerating the allowed parameters avoids a discovery scan and stabilizes schema.
PARAMETERS = ["item_id", "referrer", "viewed_user_id", "test_id", "test_assignment"]
unknown_parameters = event_parameters.filter(~F.col("parameter_name").isin(PARAMETERS)).count()
assert unknown_parameters == 0, "New parameter name found: update Silver schema and pivot."

# Grouping on event metadata and pivoting yields one fact per event.
silver_events_df = (
    event_parameters
    .groupBy("event_id", "event_time", "user_id", "event_name", "platform")
    .pivot("parameter_name", PARAMETERS)
    .agg(F.first("parameter_value", ignorenulls=True))
    .withColumn("item_id", F.col("item_id").cast("bigint"))
    .withColumn("viewed_user_id", F.col("viewed_user_id").cast("bigint"))
    .withColumn("test_id", F.col("test_id").cast("int"))
    .withColumn("test_assignment", F.col("test_assignment").cast("int"))
    .withColumn("event_year", F.year("event_time"))
    .select(
        "event_id", "event_time", "user_id", "event_name", "platform",
        "item_id", "referrer", "viewed_user_id", "test_id", "test_assignment", "event_year"
    )
)

# Validate grain and representative business activity.
assert silver_events_df.count() == 428_364
assert silver_events_df.select("event_id").distinct().count() == 428_364
assert silver_events_df.filter(F.col("event_name") == "view_item").count() == 262_786
assert silver_events_df.filter((F.col("event_name") == "view_item") & F.col("item_id").isNull()).count() == 0

silver_events_df.printSchema()
display(silver_events_df.filter(F.col("event_name") == "view_item").limit(5))

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- item_id: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- viewed_user_id: long (nullable = true)
 |-- test_id: integer (nullable = true)
 |-- test_assignment: integer (nullable = true)
 |-- event_year: integer (nullable = true)



event_id,event_time,user_id,event_name,platform,item_id,referrer,viewed_user_id,test_id,test_assignment,event_year
bde0190429054d8daed8ef4571b8c626,2017-04-08T01:10:23.000Z,185468,view_item,web,771,item_page,null,null,null,2017
c20929d2c8da455eb7a88d98dfc7e035,2015-12-31T00:18:47.000Z,95287,view_item,web,1603,google_search,null,null,null,2015
de55dce81dad4749b22ba0c4af6faa2b,2017-08-18T10:12:20.000Z,226006,view_item,iOS,67,shopping_cart,null,null,null,2017
3d17cef7150d413fae17fb884d206383,2016-11-07T03:17:20.000Z,134454,view_item,web,1273,promo_email_click,null,null,null,2016
d0be5a7af54e4b34a2bd9d82aa654015,2013-11-11T04:41:57.000Z,5948,view_item,web,1994,user_wishlist,null,null,null,2013


### 6.1 Silver fact DDL — year-partitioned Delta table

The assignment asks for meaningful fact-table partitioning. We use `event_year` because event data spans 2013–2018 and the target metrics are calculated **by year**. This allows partitions to be pruned when querying a selected year.

**Trade-off:** At this data volume, Databricks often performs well without manually specified partitions. We implement them here to meet the exercise requirement and make the partition strategy explicit. The table is partitioned by a low-cardinality column, not by `event_id` or `item_id` (which would create excessive tiny partitions).

In [0]:
# Typed event fact DDL, with physical partitioning by event_year.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_EVENT} (
    event_id STRING,
    event_time TIMESTAMP,
    user_id BIGINT,
    event_name STRING,
    platform STRING,
    item_id BIGINT,
    referrer STRING,
    viewed_user_id BIGINT,
    test_id INT,
    test_assignment INT,
    event_year INT
)
USING DELTA
PARTITIONED BY (event_year)
""")

silver_events_df.createOrReplaceTempView("stg_silver_events")

# Full load of Silver events; the partition column must be included in SELECT.
spark.sql(f"""
INSERT OVERWRITE TABLE {SILVER_EVENT}
SELECT
    event_id, event_time, user_id, event_name, platform,
    item_id, referrer, viewed_user_id, test_id, test_assignment, event_year
FROM stg_silver_events
""")

silver_event_table = spark.table(SILVER_EVENT)
assert silver_event_table.count() == 428_364
assert spark.table(SILVER_ITEM).count() == 2_198

# Check physical Delta partitioning, not just the presence of an event_year column.
partition_columns = (
    spark.sql(f"DESCRIBE DETAIL {SILVER_EVENT}")
    .select("partitionColumns")
    .first()["partitionColumns"]
)
assert partition_columns == ["event_year"]
print("Silver loaded and validated; event partitions:", partition_columns)

Silver loaded and validated; event partitions: ['event_year']


## 7. Gold — `top_item` datamart: business definitions

### 7.1 Grain, scope and important decisions

**One output row per (`item_id`, `year`).** Here, `year` means a calendar year in which there was at least one item-view event somewhere in the source dataset (2013–2018). The datamart includes **all catalog items for each such year**, even if an item was never viewed that year.

- **`total_views`:** Count of distinct logical `view_item` events for that item in that year. Because Silver has one record per unique `event_id`, `COUNT(*)` over filtered Silver item-view events is valid.
- **`item_rank`:** Descending **`DENSE_RANK()`** of `total_views` within the given year. Multiple items may share rank 1 or another rank if their totals are tied; there are no gaps in subsequent ranks.
- **`most_used_platform`:** The platform with the highest event count for that item and year. Ties are broken deterministically by ascending platform string to yield exactly one platform. Null for item-year rows with zero views.
- **Missing views:** Generate all `item × year` combinations, left join view aggregates, and use `0` as the view count where no view events exist.
- **Not used:** `test_assignment` and `view_user_profile` events do not count as item views. `referrer` is not needed in the Gold measures, but remains available in Silver.

### 7.2 Datamart data dictionary

| Column | Type | Description / rule |
|:--|:--|:--|
| `item_id` | BIGINT | Unique item identifier (part of the natural key). |
| `item_name` | STRING | Human-readable label from the item dimension. |
| `category` | STRING | Item category from the item dimension. |
| `year` | INT | Calendar year of item-view events (part of the natural key). |
| `total_views` | BIGINT | Number of logical item-view events; `0` when there are none. |
| `item_rank` | INT | `DENSE_RANK` within the year by descending total views, including zero-view items. |
| `most_used_platform` | STRING (nullable) | Most-viewed platform for that item/year; alphabetical tie-break; `NULL` for zero views. |

The metric is **event count**, not distinct viewers/users. One user viewing the same item multiple times contributes multiple events, as long as those are different `event_id` values.

**Reporting-scope caveat:** As defined, the item/year scaffold includes items before their `created_at` date, assigning zero views. If a stakeholder defines “eligible item in a year” as an item already launched at that time, filter the scaffold by the creation date instead; this notebook follows the literal “every item and year” interpretation used in the exercise.

### 7.3 Filter genuine view facts and verify item references

All item IDs referenced by `view_item` events should be present in the catalog item dimension. This is an important **referential integrity** check; otherwise joining to item descriptions can accidentally lose views.

In [0]:
# Read the persistent Silver tables; Gold never reads CSV/Bronze directly.
events = spark.table(SILVER_EVENT)
items = spark.table(SILVER_ITEM)

# Silver grain is one row per logical event, so a simple count is correct here.
views = (
    events
    .filter(F.col("event_name") == "view_item")
    .filter(F.col("item_id").isNotNull() & F.col("event_year").isNotNull())
    .select("event_id", "item_id", "event_year", "platform")
)

# Check for orphan item IDs before building the analytical mart.
missing_item_ids = (
    views.select("item_id").distinct()
    .join(items.select("item_id"), on="item_id", how="left_anti")
)
assert missing_item_ids.count() == 0, "View events reference unknown items."
assert views.count() == 262_786
print("Validated logical item views:", views.count())

Validated logical item views: 262786


### 7.4 Aggregate annual item views

The output of this step has grain (`item_id`, `event_year`) and contains one numeric metric, `total_views`.

In [0]:
# COUNT(*) is valid because Step 6 normalized one row per event_id.
yearly_views = (
    views
    .groupBy("item_id", "event_year")
    .agg(F.count("*").alias("total_views"))
)

display(yearly_views.orderBy(F.desc("total_views")).limit(10))

item_id,event_year,total_views
2609,2017,58
3806,2017,57
2425,2017,56
1978,2017,55
3531,2017,54
1064,2017,54
1804,2017,53
694,2017,53
730,2017,53
3981,2017,52


### 7.5 Calculate the most-used platform

First count events per (`item_id`, `event_year`, `platform`). Then apply a window ordered by:

1. **`platform_views DESC`** — highest event count wins;
2. **`platform ASC`** — deterministic tie-break when counts are equal.

`ROW_NUMBER = 1` returns a single platform per item/year. We deliberately exclude null platform values rather than treating missing platform as an actual channel.

In [0]:
# Platform-level aggregation: one row per item/year/platform.
platform_counts = (
    views.filter(F.col("platform").isNotNull())
    .groupBy("item_id", "event_year", "platform")
    .agg(F.count("*").alias("platform_views"))
)

platform_window = (
    Window.partitionBy("item_id", "event_year")
    .orderBy(F.desc("platform_views"), F.asc("platform"))
)

# Pick the winning platform deterministically.
top_platform = (
    platform_counts
    .withColumn("platform_row", F.row_number().over(platform_window))
    .filter(F.col("platform_row") == 1)
    .select("item_id", "event_year", F.col("platform").alias("most_used_platform"))
)

display(top_platform.limit(10))

item_id,event_year,most_used_platform
0,2013,mobile web
0,2014,web
0,2015,web
0,2016,iOS
0,2017,web
0,2018,web
1,2013,web
1,2014,web
1,2015,web
1,2016,web


### 7.6 Include zero-view item/year combinations

A `CROSS JOIN` between the 2,198-item dimension and six observed item-view years produces **13,188** potential item/year combinations. In the supplied dataset, **13,093 combinations have at least one view**; **95 have no views**.

We use **LEFT JOIN**, not inner join, for annual counts and platform information, preserving those 95 zero-view combinations. This matches the requested **every item and year** interpretation.

In [0]:
# Data-driven year dimension: calendar years observed in item-view events.
view_years = views.select("event_year").distinct()

# This crossJoin is intentionally small: 2,198 items × 6 years.
item_years = (
    items.select("item_id", "item_name", "category")
    .crossJoin(view_years)
)

# Preserve every item/year and represent missing counts as zero.
gold_base = (
    item_years
    .join(yearly_views, on=["item_id", "event_year"], how="left")
    .join(top_platform, on=["item_id", "event_year"], how="left")
    .withColumn("total_views", F.coalesce(F.col("total_views"), F.lit(0).cast("bigint")))
)

assert item_years.count() == 13_188
assert gold_base.count() == 13_188
assert gold_base.filter(F.col("total_views") == 0).count() == 95
print("All item/year combinations:", gold_base.count())

All item/year combinations: 13188


### 7.7 Annual ranking (`DENSE_RANK`)

Partition the ranking window by year, order by `total_views` descending, and use `dense_rank()`.

**Important:** The real dataset includes ties for first place: there are **two** rank-1 items in 2013, **two** in 2014 and **four** in 2015. Using `row_number()` would incorrectly assign different positions to equally viewed items. We keep the ties in the final Gold output.

In [0]:
# Ranking resets for each year and preserves ties.
item_rank_window = (
    Window.partitionBy("event_year")
    .orderBy(F.desc("total_views"))
)

gold_df = (
    gold_base
    .withColumn("item_rank", F.dense_rank().over(item_rank_window))
    .select(
        "item_id", "item_name", "category",
        F.col("event_year").alias("year"),
        F.col("total_views").cast("bigint").alias("total_views"),
        F.col("item_rank").cast("int").alias("item_rank"),
        "most_used_platform"
    )
)

display(gold_df.orderBy("year", "item_rank", "item_id").limit(20))

item_id,item_name,category,year,total_views,item_rank,most_used_platform
2086,organic device,device,2013,12,1,web
3517,prize-winning widget wrapper,widget,2013,12,1,web
3063,fuzzy gadget wrapper,gadget,2013,11,2,web
3200,prize-winning widget refill,widget,2013,10,3,web
3309,industrial-strength dongle,dongle,2013,10,3,mobile web
3538,organic dongle,dongle,2013,10,3,web
3978,fuzzy dongle,dongle,2013,10,3,android
80,rechargable module cleaner,module,2013,9,4,mobile web
138,matte widget refill,widget,2013,9,4,web
265,glossy dongle,dongle,2013,9,4,web


### 7.8 Gold DDL and datamart materialization

The final managed Delta table is intended for direct BI/reporting consumption. At about 13K records, it does **not** need manual physical partitioning. The DDL declares the fields and numeric types; the `INSERT OVERWRITE` creates a reproducible full-refresh mart.

In [0]:
# Explicit Gold table DDL.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TOP_ITEM} (
    item_id BIGINT,
    item_name STRING,
    category STRING,
    year INT,
    total_views BIGINT,
    item_rank INT,
    most_used_platform STRING
)
USING DELTA
""")

gold_df.createOrReplaceTempView("stg_gold_top_item")

# Full-refresh datamart DML; deterministic results on repeated runs.
spark.sql(f"""
INSERT OVERWRITE TABLE {GOLD_TOP_ITEM}
SELECT
    item_id, item_name, category, year,
    total_views, item_rank, most_used_platform
FROM stg_gold_top_item
""")

print("Gold datamart materialized:", GOLD_TOP_ITEM)

Gold datamart materialized: workspace.gold.top_item


## 8. End-to-end quality checks

A technically successful write is not enough: validate **volume, grain, numerical integrity, reference integrity, winner selection and physical partitioning**.

The supplied input files imply these reference counts:

| Check | Expected |
|:--|--:|
| Bronze event parameter rows | 853,640 |
| Bronze/Silver item records | 2,198 |
| Silver unique events | 428,364 |
| Silver logical item-view events | 262,786 |
| Gold distinct item/year pairs | 13,188 |
| Gold zero-view item/year pairs | 95 |
| Sum of Gold item views | 262,786 |

In [0]:
# Work against saved tables to validate persisted outputs (not only DataFrames).
bronze_event_final = spark.table(BRONZE_EVENT)
bronze_item_final = spark.table(BRONZE_ITEM)
silver_event_final = spark.table(SILVER_EVENT)
silver_item_final = spark.table(SILVER_ITEM)
gold_final = spark.table(GOLD_TOP_ITEM)

# Counts and keys.
assert bronze_event_final.count() == 853_640
assert bronze_item_final.count() == 2_198
assert silver_event_final.count() == 428_364
assert silver_item_final.count() == 2_198
assert gold_final.count() == 13_188
assert gold_final.select("item_id", "year").distinct().count() == 13_188
assert gold_final.filter(F.col("total_views") == 0).count() == 95

# Measure conservation: Silver and Gold must count the same number of views.
gold_view_count = gold_final.agg(F.sum("total_views").alias("n")).first()["n"]
assert gold_view_count == 262_786
assert gold_view_count == silver_event_final.filter(F.col("event_name") == "view_item").count()

# First-place ties are intentional: DENSE_RANK must not discard tied winners.
rank_one_counts = {row["year"]: row["n"] for row in (
    gold_final.filter(F.col("item_rank") == 1)
    .groupBy("year").agg(F.count("*").alias("n")).collect()
)}
assert rank_one_counts == {2013: 2, 2014: 2, 2015: 4, 2016: 1, 2017: 1, 2018: 1}

# Every viewed item has a platform; items with zero views have no platform.
assert gold_final.filter((F.col("total_views") > 0) & F.col("most_used_platform").isNull()).count() == 0
assert gold_final.filter((F.col("total_views") == 0) & F.col("most_used_platform").isNotNull()).count() == 0

# No invalid ranks, negative counts or missing fact keys.
assert gold_final.filter(
    F.col("item_id").isNull() | F.col("year").isNull() |
    (F.col("total_views") < 0) | (F.col("item_rank") < 1)
).count() == 0

print("PASS — source retention, fact grain, item-year keys, view counts and platform coverage")
print("Gold rows:", gold_final.count(), "| total views:", gold_view_count)

PASS — source retention, fact grain, item-year keys, view counts and platform coverage
Gold rows: 13188 | total views: 262786


### 8.1 Inspect all first-place items (including ties)

This diagnostic query intentionally returns **more than one item per year when items tie for first place**. It is useful for review by business stakeholders.

In [0]:
# Datamart query written in SQL for analysts who don't use PySpark.
# Sorting makes results easy to compare across notebook runs.
spark.sql(f"""
SELECT year, item_id, item_name, total_views, item_rank, most_used_platform
FROM {GOLD_TOP_ITEM}
WHERE item_rank = 1
ORDER BY year, item_id
""").show(50, truncate=False)

+----+-------+----------------------------------------+-----------+---------+------------------+
|year|item_id|item_name                               |total_views|item_rank|most_used_platform|
+----+-------+----------------------------------------+-----------+---------+------------------+
|2013|2086   |organic device                          |12         |1        |web               |
|2013|3517   |prize-winning widget wrapper            |12         |1        |web               |
|2014|244    |miniature contraption                   |29         |1        |web               |
|2014|1353   |mechanism                               |29         |1        |web               |
|2015|55     |prize-winning instrument charger        |39         |1        |web               |
|2015|511    |extra-strength mechanism storage_unit   |39         |1        |web               |
|2015|1327   |extra-strength apparatus charger        |39         |1        |web               |
|2015|2094   |analog tool     

### 8.2 Additional business SQL examples

The Gold layer hides the raw JSON, event-parameter pivot logic, and platform-specific windowing from end users. Analysts can now ask straightforward questions:

- What are the 10 most-viewed items in 2017?
- How many views are recorded per year?
- Which platforms are most frequently the winning platform across the item catalog?

In [0]:
# Example 1: top ten items in 2017, with stable sorting for tied ranks.
display(spark.sql(f"""
SELECT year, item_id, item_name, total_views, item_rank, most_used_platform
FROM {GOLD_TOP_ITEM}
WHERE year = 2017
ORDER BY item_rank, item_id
LIMIT 10
"""))

# Example 2: yearly views across all items.
display(spark.sql(f"""
SELECT year, SUM(total_views) AS total_item_views
FROM {GOLD_TOP_ITEM}
GROUP BY year
ORDER BY year
"""))

year,item_id,item_name,total_views,item_rank,most_used_platform
2017,2609,extra-strength contraption how-to-manual,58,1,web
2017,3806,extra-strength device opener,57,2,web
2017,2425,apparatus,56,3,web
2017,1978,aerodynamic device wrapper,55,4,web
2017,1064,aerodynamic contraption warmer,54,5,web
2017,3531,industrial-strength device,54,5,web
2017,694,miniature widget,53,6,web
2017,730,organic apparatus carrying_case,53,6,web
2017,1804,prize-winning apparatus cleaner,53,6,web
2017,765,miniature module,52,7,web


year,total_item_views
2013,7448
2014,32082
2015,49462
2016,63642
2017,76314
2018,33838


## 9. Engineering decisions, limitations and possible improvements

| Decision | Why it was chosen | Production consideration |
|:--|:--|:--|
| Raw attributes stored as STRING | Strict Bronze requirement; protects fidelity and reproducibility. | Capture ingest time, source-file name and run ID in an **additional** technical audit table if permitted; not added here because the assignment requires no raw transformations. |
| Explicit CSV escape character | Doubled double quotes are used inside the source JSON string. | Add input-contract/schema drift checks and quarantine malformed CSVs. |
| `from_json` with declared schema | Avoids repeated extraction and provides typed access to nested fields. | Maintain a versioned payload schema when upstream JSON evolves. |
| Pivot source parameters to one event row | Prevents double-counting while retaining event attributes. | Detect and resolve duplicated/conflicting parameters; this notebook fails before pivoting if duplicates occur. |
| `event_year` partitioning | Low-cardinality year filter relevant to yearly aggregations; required by assignment. | Small Delta tables often benefit from no manual partitions; benchmark before using this in production. |
| `item × year` scaffold | Includes items with zero views, matching “every item and year.” | If reporting requires only active items/years, remove the cross join and use observed item-year aggregates instead. |
| `dense_rank` | Equal views receive equal rank, with no skipped ranks. | Agree ranking semantics with business; `RANK()` is another valid convention. |
| Alphabetical platform tie-break | Deterministic winner when usage counts are equal. | A business priority order or multi-platform output could be more appropriate. |
| Full-table `INSERT OVERWRITE` | Repeatable, simple batch case-study execution. | For regularly arriving data, use incremental processing, checkpointing, `MERGE`, job orchestration and operational monitoring. |

**Timestamp assumption:** Source timestamps do not include an explicit timezone. This notebook uses Spark's session interpretation for `to_timestamp`; production pipelines should establish the upstream timezone and standardize it (often to UTC).

**Price assumption:** Prices are interpreted as monetary amounts with two decimals; if sub-cent precision or a specific currency is required, confirm this with business stakeholders.

**Reconciliation note:** The fixture has 2,198 unique item IDs, 262,786 valid item-view events, no unknown item references, and 95 item/year combinations with no view. These are assertions for the **supplied case-study files**, not universal expectations for evolving future input.

---

### Deliverables produced by this notebook

- `workspace.bronze.item`, `workspace.bronze.event` — raw Delta tables, all `STRING`.
- `workspace.silver.item`, `workspace.silver.event` — typed, flattened tables; fact table partitioned by year.
- `workspace.gold.top_item` — documented annual datamart.
- Validation assertions and SQL query examples.